# 09 · Validación con la muestra etiquetada a mano (subconjunto "oro")

Las etiquetas de entrenamiento se generaron con **reglas**. Un riesgo es que los modelos simplemente **aprendan las reglas**
(las mismas palabras clave) y por eso obtengan métricas altas contra esas mismas etiquetas. Para medir la calidad real se usa
una muestra de la partición de **prueba** etiquetada **a mano**:

1. **Acuerdo reglas vs. humano:** accuracy y **kappa de Cohen** (acuerdo corregido por azar).
2. **Desempeño de los 4 modelos contra la etiqueta humana** (la "verdad" más confiable).

### Cómo usarlo
1. Abrir `datos/muestra_validacion.csv` y escribir en la columna `etiqueta_manual` una de: `Producto`, `Envío` o `Sentimiento`
   (también se aceptan `P`, `E`, `S`, `envio`, `producto`, `sentimiento`). Guardar como CSV (UTF-8).
2. Volver a ejecutar este notebook completo (*Kernel → Restart & Run All*).

Se pueden dejar filas vacías: solo se usan las que tienen etiqueta. Mientras el archivo esté vacío, el notebook solo muestra estas instrucciones.

> Nota: la muestra está **estratificada por la etiqueta de las reglas** (hasta 150 por clase), no sigue la distribución natural.
> Por eso la accuracy de esta muestra no es comparable directamente con la del conjunto de prueba completo; el **F1 macro** y
> el **kappa** son las métricas más informativas aquí.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, classification_report, confusion_matrix

import utils
from utils import CLASES, RESULTADOS, MODELOS

SALIDA = RESULTADOS / "validacion_manual"
SALIDA.mkdir(parents=True, exist_ok=True)
# Se puede indicar otro archivo con la variable de entorno MUESTRA_VALIDACION (útil para pruebas)
RUTA_MUESTRA = Path(os.environ.get("MUESTRA_VALIDACION", utils.ARCHIVO_VALIDACION))

muestra = pd.read_csv(RUTA_MUESTRA, encoding="utf-8-sig", keep_default_na=False)
muestra["etiqueta_manual_norm"] = muestra["etiqueta_manual"].apply(utils.normalizar_etiqueta_manual)
escritas = muestra["etiqueta_manual"].astype(str).str.strip() != ""
invalidas = muestra[escritas & muestra["etiqueta_manual_norm"].isna()]
llenas = muestra[muestra["etiqueta_manual_norm"].notna()].copy()
HAY_DATOS = len(llenas) > 0

print(f"Archivo: {RUTA_MUESTRA.name} | reseñas en la muestra: {len(muestra)} | con etiqueta manual válida: {len(llenas)}")
if len(invalidas):
    print(f"AVISO: {len(invalidas)} etiquetas no reconocidas (se ignoran):", list(invalidas["etiqueta_manual"].unique()[:10]))
if not HAY_DATOS:
    print("\nTodavía no hay etiquetas manuales. Llena la columna 'etiqueta_manual' de datos/muestra_validacion.csv "
          "con Producto / Envío / Sentimiento y vuelve a ejecutar este notebook.")
elif len(llenas) < 100:
    print("AVISO: con menos de 100 reseñas etiquetadas los resultados serán poco estables.")

Archivo: muestra_validacion.csv | reseñas en la muestra: 368 | con etiqueta manual válida: 0

Todavía no hay etiquetas manuales. Llena la columna 'etiqueta_manual' de datos/muestra_validacion.csv con Producto / Envío / Sentimiento y vuelve a ejecutar este notebook.


## 1. Acuerdo entre las reglas y la etiqueta manual

**Interpretación del kappa de Cohen** (Landis y Koch, 1977): < 0.20 pobre · 0.21-0.40 débil · 0.41-0.60 moderado ·
0.61-0.80 sustancial · > 0.80 casi perfecto.

In [2]:
def interpretar_kappa(k):
    return ("pobre" if k <= 0.20 else "débil" if k <= 0.40 else "moderado" if k <= 0.60
            else "sustancial" if k <= 0.80 else "casi perfecto")

if HAY_DATOS:
    procesado = pd.read_csv(utils.ARCHIVO_PROCESADO, encoding="utf-8-sig", keep_default_na=False)
    oro = llenas.merge(procesado[["id", "etiqueta_reglas", "texto_procesado", "polarity", "particion"]], on="id", how="left")
    assert oro["etiqueta_reglas"].notna().all(), "Hay ids de la muestra que no están en el corpus procesado"
    assert (oro["particion"] == "prueba").all(), "La muestra debe venir solo de la partición de prueba"
    y_manual, y_reglas = oro["etiqueta_manual_norm"], oro["etiqueta_reglas"]

    acuerdo = accuracy_score(y_manual, y_reglas)
    kappa = cohen_kappa_score(y_manual, y_reglas)
    print(f"Reseñas evaluadas: {len(oro)}")
    print("Distribución de la etiqueta manual:", y_manual.value_counts().reindex(CLASES).fillna(0).astype(int).to_dict())
    print(f"Acuerdo (accuracy) reglas vs. manual: {acuerdo:.4f}")
    print(f"Kappa de Cohen: {kappa:.4f}  ->  acuerdo {interpretar_kappa(kappa)}")
    print("\nPrecisión/recall de las REGLAS tomando la etiqueta manual como verdad:")
    print(classification_report(y_manual, y_reglas, labels=CLASES, digits=4, zero_division=0))

    matriz = confusion_matrix(y_manual, y_reglas, labels=CLASES)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Purples", cbar=False, xticklabels=CLASES, yticklabels=CLASES, ax=ax)
    ax.set_xlabel("Etiqueta de las reglas"); ax.set_ylabel("Etiqueta manual"); ax.set_title("Reglas vs. etiquetado manual")
    fig.tight_layout(); fig.savefig(SALIDA / "matriz_reglas_vs_manual.png", dpi=120); plt.show()
    pd.DataFrame(matriz, index=[f"manual_{c}" for c in CLASES], columns=[f"reglas_{c}" for c in CLASES]) \
        .to_csv(SALIDA / "matriz_reglas_vs_manual.csv", encoding="utf-8-sig")
else:
    print("Sin etiquetas manuales todavía: se omite esta sección.")

Sin etiquetas manuales todavía: se omite esta sección.


## 2. Los 4 modelos evaluados contra la etiqueta manual

Para cada modelo guardado se calcula su desempeño contra la etiqueta **manual** y, en la misma muestra, contra la etiqueta de
las **reglas**. Si las métricas contra las reglas son mucho más altas que contra el humano, el modelo está reproduciendo las
reglas más que "entendiendo" las reseñas.

In [3]:
if HAY_DATOS:
    filas = []
    for nombre, clave in utils.MODELOS_PROYECTO.items():
        modelo = joblib.load(MODELOS / f"{clave}.joblib")
        pred = modelo.predict(oro["texto_procesado"])
        filas.append({
            "Modelo": nombre,
            "Accuracy vs manual": accuracy_score(y_manual, pred),
            "F1 macro vs manual": f1_score(y_manual, pred, labels=CLASES, average="macro", zero_division=0),
            "Kappa vs manual": cohen_kappa_score(y_manual, pred),
            "F1 Envío vs manual": f1_score(y_manual, pred, labels=["Envío"], average="macro", zero_division=0),
            "F1 macro vs reglas (misma muestra)": f1_score(y_reglas, pred, labels=CLASES, average="macro", zero_division=0),
        })
    filas.append({"Modelo": "Reglas (referencia)", "Accuracy vs manual": acuerdo,
                  "F1 macro vs manual": f1_score(y_manual, y_reglas, labels=CLASES, average="macro", zero_division=0),
                  "Kappa vs manual": kappa,
                  "F1 Envío vs manual": f1_score(y_manual, y_reglas, labels=["Envío"], average="macro", zero_division=0),
                  "F1 macro vs reglas (misma muestra)": 1.0})
    validacion = pd.DataFrame(filas).set_index("Modelo")
    validacion.to_csv(SALIDA / "modelos_vs_manual.csv", encoding="utf-8-sig")
    display(validacion.round(4))

    fig, ax = plt.subplots(figsize=(9, 4.5))
    validacion[["F1 macro vs manual", "F1 macro vs reglas (misma muestra)"]].plot(kind="bar", ax=ax, rot=0)
    ax.set_ylim(0, 1.05); ax.set_title("F1 macro: contra etiqueta manual vs. contra reglas")
    for contenedor in ax.containers:
        ax.bar_label(contenedor, fmt="%.2f", fontsize=7)
    fig.tight_layout(); fig.savefig(SALIDA / "modelos_vs_manual.png", dpi=120); plt.show()
    mejor = validacion.drop(index="Reglas (referencia)")["F1 macro vs manual"].idxmax()
    print(f"Modelo con mayor F1 macro contra la etiqueta manual: {mejor}")
else:
    print("Sin etiquetas manuales todavía: se omite esta sección.")

Sin etiquetas manuales todavía: se omite esta sección.
